In [ ]:
!pip -q install lightgbm==4.6.0
import os, json, time, gc, warnings
warnings.filterwarnings('ignore', message='X does not have valid feature names')
import numpy as np, pandas as pd, lightgbm as lgb
from sklearn.metrics import average_precision_score
try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    pass
print('pandas', pd.__version__, '| numpy', np.__version__, '| lightgbm', lgb.__version__)

In [ ]:
SMOKE = True                      # True: UNSW-v2, seed 42, 300 rounds. False: full run.

DATA_DIR = '/content/drive/MyDrive'
OUT_DIR  = '/content/drive/MyDrive/THAT_IDS_RERUN/journal_step20_unseen_vector_v1'

DATASETS = {
    'UNSW_V2': dict(file='NF-UNSW-NB15-v2.csv', rows=2_390_275, groups=236_999,
                    overlap={42: 0.93465, 123: 0.93465, 456: 0.93497}),
    'CIC_V2':  dict(file='NF-CSE-CIC-IDS2018-v2.csv', rows=18_893_708, groups=8_981_062,
                    overlap={42: 0.54824, 123: 0.54790, 456: 0.54817}),
}
SEEDS = [42, 123, 456]
EXCLUDE_COLS = ['FLOW_START_MILLISECONDS', 'FLOW_END_MILLISECONDS', 'IPV4_SRC_ADDR',
                'IPV4_DST_ADDR', 'L4_SRC_PORT', 'L4_DST_PORT', 'Label', 'Attack']
N_FEATURES = 39
CHUNK = 1_000_000

LGBM_PARAMS = dict(objective='binary', n_estimators=2000, learning_rate=0.03, num_leaves=31,
                   max_depth=-1, min_child_samples=50, subsample=0.8, subsample_freq=1,
                   colsample_bytree=0.8, reg_alpha=0.0, reg_lambda=1.0, max_bin=255,
                   deterministic=True, force_col_wise=True, n_jobs=-1, verbose=-1)
EARLY_STOP = 100

if SMOKE:
    RUN = [('UNSW_V2', 42)]
    LGBM_PARAMS['n_estimators'] = 300
    OUT_DIR = OUT_DIR + '_smoke'
else:
    RUN = [(d, s) for d in DATASETS for s in SEEDS]
os.makedirs(OUT_DIR, exist_ok=True)
print('runs:', RUN); print('output:', OUT_DIR)

In [ ]:
def load_dataset(name):
    cfg = DATASETS[name]
    path = os.path.join(DATA_DIR, cfg['file'])
    if not os.path.exists(path):
        raise FileNotFoundError(f'{path} not found: edit DATA_DIR in the config cell')
    header = pd.read_csv(path, nrows=0).columns.tolist()
    feat_cols = [c for c in header if c not in EXCLUDE_COLS]
    assert len(feat_cols) == N_FEATURES, f'{len(feat_cols)} feature columns, expected {N_FEATURES}'
    n = cfg['rows']
    X = np.empty((n, N_FEATURES), dtype=np.float32)
    keys = np.empty(n, dtype=np.uint64)
    y = np.empty(n, dtype=np.int8)
    fam = np.empty(n, dtype=np.int16)
    fam_names, pos, t0 = {}, 0, time.time()
    for chunk in pd.read_csv(path, usecols=feat_cols + ['Label', 'Attack'], chunksize=CHUNK):
        f = chunk[feat_cols].apply(pd.to_numeric, errors='coerce').astype(np.float32)
        a = f.to_numpy(copy=True)
        a[~np.isfinite(a)] = 0.0
        f = pd.DataFrame(a, columns=feat_cols)
        m = len(f)
        if pos + m > n:
            raise ValueError(f'{name}: more rows than expected ({n})')
        X[pos:pos + m] = a
        keys[pos:pos + m] = pd.util.hash_pandas_object(f, index=False).to_numpy()
        y[pos:pos + m] = chunk['Label'].to_numpy().astype(np.int8)
        att = chunk['Attack'].astype(str)
        for v in att.unique():
            fam_names.setdefault(v, len(fam_names))
        fam[pos:pos + m] = att.map(fam_names).to_numpy().astype(np.int16)
        pos += m
        print(f'\r{name}: {pos:,} rows read ({time.time() - t0:.0f}s)', end='')
    print()
    assert pos == n, f'{name}: read {pos:,} rows, expected {n:,}'
    n_groups = len(np.unique(keys))
    print(f'{name}: groups {n_groups:,} (expected {cfg["groups"]:,}) | '
          f'repeated-row rate {100 * (1 - n_groups / n):.2f}% | attack {100 * y.mean():.2f}%')
    if n_groups != cfg['groups']:
        print('  WARNING: group count differs from the audit, so preprocessing does not match the paper.')
    names = [k for k, _ in sorted(fam_names.items(), key=lambda kv: kv[1])]
    return X, keys, y, fam, names, feat_cols, n_groups

In [ ]:
def random_row_split(fam, seed):
    # stratified by attack family; each family permuted with the same seed, cut 70/10/20
    tr, va, te = [], [], []
    for f in np.unique(fam):
        idx = np.flatnonzero(fam == f)
        idx = idx[np.random.default_rng(seed).permutation(len(idx))]
        a, b = int(0.70 * len(idx)), int(0.80 * len(idx))
        tr.append(idx[:a]); va.append(idx[a:b]); te.append(idx[b:])
    return [np.sort(np.concatenate(p)) for p in (tr, va, te)]

def best_f1_macro_threshold(y, s):
    # exact search over distinct validation scores; prediction is score >= threshold
    order = np.argsort(-s, kind='stable')
    ys, ss = y[order].astype(np.int64), s[order]
    tp, fp = np.cumsum(ys), np.cumsum(1 - ys)
    last = np.r_[ss[1:] != ss[:-1], True]
    tp, fp, thr = tp[last], fp[last], ss[last]
    P = ys.sum(); N = len(ys) - P
    fn, tn = P - tp, N - fp
    f1 = 0.5 * (2 * tp / (2 * tp + fp + fn) + 2 * tn / (2 * tn + fn + fp))
    return float(thr[np.argmax(f1)])

def metrics(y, s, pred):
    y = y.astype(np.int64); pred = pred.astype(np.int64)
    tp = int((pred & y).sum()); fp = int((pred & (1 - y)).sum())
    fn = int(((1 - pred) & y).sum()); tn = int(((1 - pred) & (1 - y)).sum())
    div = lambda a, b: a / b if b else float('nan')
    out = dict(n=len(y), n_attack=tp + fn, n_benign=fp + tn,
               attack_pct=100 * div(tp + fn, len(y)),
               f1_macro=0.5 * (div(2 * tp, 2 * tp + fp + fn) + div(2 * tn, 2 * tn + fn + fp)),
               fpr_pct=100 * div(fp, fp + tn), tpr=div(tp, tp + fn), precision=div(tp, tp + fp),
               tp=tp, fp=fp, fn=fn, tn=tn)
    out['pr_auc'] = (float(average_precision_score(y, s))
                     if s is not None and 0 < tp + fn < len(y) else float('nan'))
    return out

def family_recall(y, pred, fam, names):
    rows = []
    for f in np.unique(fam[y == 1]):
        m = (fam == f) & (y == 1)
        rows.append(dict(family=names[f], count=int(m.sum()), recall=float(pred[m].mean())))
    return rows

In [ ]:
def run_one(name, seed, data):
    X, keys, y, fam, names, feat_cols, n_groups = data
    cfg = DATASETS[name]
    tr, va, te = random_row_split(fam, seed)
    assert len(np.intersect1d(tr, te)) == 0 and len(tr) + len(va) + len(te) == len(y)

    # seen / unseen by exact feature-vector hash
    order = np.argsort(keys[tr], kind='stable')
    k_sorted, y_sorted = keys[tr][order], y[tr][order]
    pos = np.clip(np.searchsorted(k_sorted, keys[te]), 0, len(k_sorted) - 1)
    seen = k_sorted[pos] == keys[te]
    overlap = float(seen.mean())
    ref = cfg['overlap'].get(seed)
    print(f'[{name} seed {seed}] train {len(tr):,} | val {len(va):,} | test {len(te):,} '
          f'(attack {int(y[te].sum()):,}) | test overlap {overlap:.5f}'
          + (f' (audit {ref:.5f})' if ref else ''))
    if ref and abs(overlap - ref) > 1e-3:
        print('  WARNING: overlap differs from the audit by more than 0.1 pp; the split rule may differ.')

    # MinMax scaler fitted on train only
    Xtr = X[tr]; mn, mx = Xtr.min(0), Xtr.max(0)
    rng_ = np.where(mx > mn, mx - mn, 1.0).astype(np.float32)
    Xtr -= mn; Xtr /= rng_
    Xva = (X[va] - mn) / rng_
    Xte = (X[te] - mn) / rng_
    ytr, yva, yte, fte = y[tr], y[va], y[te], fam[te]

    spw = min(float((ytr == 0).sum()) / max(int((ytr == 1).sum()), 1), 12.0)
    model = lgb.LGBMClassifier(**LGBM_PARAMS, scale_pos_weight=spw, random_state=seed,
                               bagging_seed=seed, feature_fraction_seed=seed, data_random_seed=seed)
    t0 = time.time()
    model.fit(Xtr, ytr, eval_set=[(Xva, yva)], eval_metric='binary_logloss',
              callbacks=[lgb.early_stopping(EARLY_STOP, verbose=False)])
    wall = time.time() - t0
    s_va = model.predict_proba(Xva)[:, 1]
    s_te = model.predict_proba(Xte)[:, 1]
    thr = best_f1_macro_threshold(yva, s_va)
    pred = s_te >= thr

    # model-free lookup: copy the train label for seen vectors, benign otherwise
    look = np.where(seen, y_sorted[pos], 0).astype(np.int8)

    base = dict(dataset=name, seed=seed, split='random', threshold=thr,
                best_iteration=int(model.best_iteration_ or LGBM_PARAMS['n_estimators']),
                scale_pos_weight=spw, test_overlap=overlap, train_seconds=round(wall, 1), smoke=SMOKE)
    runs, fams = [], []
    for subset, m in [('all', np.ones(len(te), bool)), ('seen', seen), ('unseen', ~seen)]:
        runs.append({**base, 'model': 'lightgbm', 'subset': subset, **metrics(yte[m], s_te[m], pred[m])})
        runs.append({**base, 'model': 'lookup', 'subset': subset, **metrics(yte[m], None, look[m])})
        for r in family_recall(yte[m], pred[m], fte[m], names):
            fams.append(dict(dataset=name, seed=seed, model='lightgbm', subset=subset, **r))

    np.savez_compressed(os.path.join(OUT_DIR, f'{name}_seed{seed}_test_predictions.npz'),
                        row_index=te.astype(np.int64), score=s_te.astype(np.float32),
                        label=yte, family=fte, seen=seen, key=keys[te], family_names=np.array(names))
    with open(os.path.join(OUT_DIR, f'{name}_seed{seed}_result.json'), 'w') as fh:
        json.dump(dict(runs=runs, family_recall=fams, n_groups=n_groups,
                       feature_columns=feat_cols, lgbm_params=LGBM_PARAMS), fh, indent=1)
    show = pd.DataFrame(runs)[['model', 'subset', 'n', 'n_attack', 'attack_pct', 'f1_macro',
                               'pr_auc', 'fpr_pct', 'tpr', 'precision']]
    print(show.to_string(index=False, float_format=lambda v: f'{v:.4f}'))
    print(f'  train {wall:.0f}s, best iteration {base["best_iteration"]}, threshold {thr:.4f}\n')
    del Xtr, Xva, Xte, model; gc.collect()

for name in dict.fromkeys(d for d, _ in RUN):
    todo = [s for d, s in RUN if d == name
            and not os.path.exists(os.path.join(OUT_DIR, f'{name}_seed{s}_result.json'))]
    if not todo:
        print(f'{name}: all seeds already done, skipping'); continue
    data = load_dataset(name)
    for seed in todo:
        run_one(name, seed, data)
    del data; gc.collect()

In [ ]:
runs, fams = [], []
for fn in sorted(os.listdir(OUT_DIR)):
    if fn.endswith('_result.json'):
        r = json.load(open(os.path.join(OUT_DIR, fn)))
        runs += r['runs']; fams += r['family_recall']
runs, fams = pd.DataFrame(runs), pd.DataFrame(fams)
runs.to_csv(os.path.join(OUT_DIR, 'step20_all_runs.csv'), index=False)
fams.to_csv(os.path.join(OUT_DIR, 'step20_family_recall_runs.csv'), index=False)

cols = ['n', 'n_attack', 'attack_pct', 'f1_macro', 'pr_auc', 'fpr_pct', 'tpr', 'precision']
order = ['all', 'seen', 'unseen']
summary = (runs.groupby(['dataset', 'model', 'subset'])[cols].agg(['mean', 'std'])
               .reindex(order, level='subset'))
summary.to_csv(os.path.join(OUT_DIR, 'step20_summary.csv'))
pd.set_option('display.width', 250); pd.set_option('display.max_columns', 50)
print('seeds per dataset:', runs.groupby('dataset')['seed'].nunique().to_dict(), '\n')
print(summary.xs('mean', axis=1, level=1).round(4).to_string(), '\n')
print('sample s.d. over seeds:')
print(summary.xs('std', axis=1, level=1)[['f1_macro', 'pr_auc', 'fpr_pct', 'tpr']].round(4).to_string(), '\n')

piv = runs[runs.model == 'lightgbm'].pivot_table(index=['dataset', 'seed'], columns='subset',
                                                 values=['f1_macro', 'fpr_pct', 'tpr'])
delta = pd.DataFrame({f'{m}_unseen_minus_all': piv[m]['unseen'] - piv[m]['all']
                      for m in ['f1_macro', 'fpr_pct', 'tpr']})
delta.to_csv(os.path.join(OUT_DIR, 'step20_delta_unseen_minus_all.csv'))
print('LightGBM, unseen minus all, per seed:'); print(delta.round(4).to_string(), '\n')

fam_tab = (fams.groupby(['dataset', 'family', 'subset'])[['count', 'recall']].mean()
               .unstack('subset').reindex(columns=order, level='subset'))
fam_tab.to_csv(os.path.join(OUT_DIR, 'step20_family_recall_mean.csv'))
print('Per-family recall (mean over seeds; count is mean test support):')
print(fam_tab.round(4).to_string())

In [ ]:
p = ('/content/drive/MyDrive/THAT_IDS_RERUN/journal_step11_gonogo_hostpair_v1/paper/'
     'CIC_V2/random/lightgbm/seed_42/test_predictions.npz')
if os.path.exists(p):
    z = np.load(p, allow_pickle=True)
    for k in z.files:
        print(k, z[k].dtype, z[k].shape)
else:
    print('not found:', p)